In [ ]:
import ROOT

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTPUT_FILE = "tree_structure.txt"


In [ ]:
# ─────────────────────────────────────────────
# open file and tree
# ─────────────────────────────────────────────

f = ROOT.TFile.Open(ROOT_FILE)
t = f.Get(TREE_PATH)

print("Opened tree with", t.GetEntries(), "events")


# ─────────────────────────────────────────────
# save branch structure to file
# ─────────────────────────────────────────────

ROOT.gSystem.RedirectOutput(OUTPUT_FILE, "w")
t.Print()
ROOT.gSystem.RedirectOutput("", "")

print(f"Tree structure written to {OUTPUT_FILE}")


# ─────────────────────────────────────────────
# inspect first few events
# ─────────────────────────────────────────────

N_EVENTS = 5

print("\nInspecting first events:\n")

for i in range(min(N_EVENTS, t.GetEntries())):

    t.GetEntry(i)

    # example sim-track branches
    # adjust names after we inspect tree_structure.txt

    if hasattr(t, "sim_pt"):
        n = len(t.sim_pt)
        print(f"Event {i}: sim tracks = {n}")

        if n > 0:
            print("   first sim_pt:", t.sim_pt[0])

    if hasattr(t, "sim_eta"):
        print("   sim_eta size:", len(t.sim_eta))

    if hasattr(t, "sim_phi"):
        print("   sim_phi size:", len(t.sim_phi))

In [ ]:
#imports for the root file rip

from __future__ import annotations

import math
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import ROOT

In [ ]:
# Paralell data rip, splits up into n-chunks then writes each to a npz file then we merge

# we implement some filters like the naive eta limitation etc

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTDIR = Path("track_chunks_etaM1to1")
FINAL_OUTFILE = "track_data_etaM1to1.npz"

ETA_MIN = -1.0
ETA_MAX = 1.0
REQUIRE_NVALID_POSITIVE = True
REQUIRE_CHARGED = True

N_WORKERS = 20

FEATURE_BRANCHES = [
    "sim_q",
    "sim_pdgId",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
    "sim_nPixelLay",
    "sim_n3DLay",
    "sim_nTrackerHits",
    "sim_nRecoClusters",
]

LABEL_BRANCHES = [
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

# These are only for filtering / sanity checks
FILTER_BRANCHES = [
    "sim_q",
    "sim_nValid",
    "sim_pca_eta",
]


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def vec_to_np(v, dtype=np.float32):
    return np.asarray(list(v), dtype=dtype)


def all_same_length(named_arrays: dict[str, np.ndarray]) -> bool:
    if not named_arrays:
        return True
    lengths = [len(a) for a in named_arrays.values()]
    return len(set(lengths)) == 1


def split_ranges(n_events: int, n_chunks: int) -> list[tuple[int, int, int]]:
    ranges = []
    chunk_size = math.ceil(n_events / n_chunks)
    for chunk_id in range(n_chunks):
        start = chunk_id * chunk_size
        end = min((chunk_id + 1) * chunk_size, n_events)
        if start < end:
            ranges.append((chunk_id, start, end))
    return ranges


# ─────────────────────────────────────────────
# Worker
# ─────────────────────────────────────────────

def process_chunk(chunk_id: int, evt_start: int, evt_end: int, outdir_str: str):
    outdir = Path(outdir_str)
    outdir.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"[chunk {chunk_id}] Could not open ROOT file")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"[chunk {chunk_id}] Could not get tree {TREE_PATH}")

    feature_chunks = []
    label_chunks = []

    n_total_rows = 0
    n_kept_rows = 0
    n_bad_length_events = 0
    n_empty_events = 0

    for i_evt in range(evt_start, evt_end):
        t.GetEntry(i_evt)

        feat_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in FEATURE_BRANCHES
        }
        lab_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in LABEL_BRANCHES
        }
        filt_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in FILTER_BRANCHES
        }

        all_dict = {}
        all_dict.update(feat_dict)
        all_dict.update(lab_dict)
        all_dict.update(filt_dict)

        if not all_same_length(all_dict):
            n_bad_length_events += 1
            continue

        n = len(next(iter(all_dict.values())))
        if n == 0:
            n_empty_events += 1
            continue

        n_total_rows += n

        X_evt = np.column_stack([feat_dict[name] for name in FEATURE_BRANCHES]).astype(np.float32, copy=False)
        Y_evt = np.column_stack([lab_dict[name] for name in LABEL_BRANCHES]).astype(np.float32, copy=False)

        sim_q = filt_dict["sim_q"]
        sim_nValid = filt_dict["sim_nValid"]
        sim_pca_eta = filt_dict["sim_pca_eta"]

        mask = np.ones(n, dtype=bool)

        # finite checks
        mask &= np.all(np.isfinite(X_evt), axis=1)
        mask &= np.all(np.isfinite(Y_evt), axis=1)
        mask &= np.isfinite(sim_q)
        mask &= np.isfinite(sim_nValid)
        mask &= np.isfinite(sim_pca_eta)

        # eta cut
        mask &= (sim_pca_eta >= ETA_MIN) & (sim_pca_eta <= ETA_MAX)

        # charged only
        if REQUIRE_CHARGED:
            mask &= (sim_q != 0)

        # nonzero/valid-hit requirement
        if REQUIRE_NVALID_POSITIVE:
            mask &= (sim_nValid > 0)

        X_evt = X_evt[mask]
        Y_evt = Y_evt[mask]

        if len(X_evt) > 0:
            feature_chunks.append(X_evt)
            label_chunks.append(Y_evt)
            n_kept_rows += len(X_evt)

    if feature_chunks:
        X = np.concatenate(feature_chunks, axis=0)
        Y = np.concatenate(label_chunks, axis=0)
    else:
        X = np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
        Y = np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    outfile = outdir / f"chunk_{chunk_id:03d}.npz"
    np.savez_compressed(
        outfile,
        X=X,
        Y=Y,
        evt_start=evt_start,
        evt_end=evt_end,
        n_total_rows=n_total_rows,
        n_kept_rows=n_kept_rows,
        n_bad_length_events=n_bad_length_events,
        n_empty_events=n_empty_events,
    )

    return {
        "chunk_id": chunk_id,
        "evt_start": evt_start,
        "evt_end": evt_end,
        "outfile": str(outfile),
        "shape_X": X.shape,
        "shape_Y": Y.shape,
        "n_total_rows": n_total_rows,
        "n_kept_rows": n_kept_rows,
        "n_bad_length_events": n_bad_length_events,
        "n_empty_events": n_empty_events,
    }


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"Could not open ROOT file: {ROOT_FILE}")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"Could not get tree: {TREE_PATH}")

    n_events = t.GetEntries()
    print(f"Opened tree with {n_events} events")

    ranges = split_ranges(n_events, N_WORKERS)
    print(f"Using {len(ranges)} worker chunks")

    results = []
    with ProcessPoolExecutor(max_workers=len(ranges)) as ex:
        futures = [
            ex.submit(process_chunk, chunk_id, start, end, str(OUTDIR))
            for chunk_id, start, end in ranges
        ]

        for fut in as_completed(futures):
            res = fut.result()
            results.append(res)
            print(
                f"[done] chunk {res['chunk_id']:03d} "
                f"events {res['evt_start']}:{res['evt_end']} "
                f"X{res['shape_X']} Y{res['shape_Y']} "
                f"kept {res['n_kept_rows']:,}/{res['n_total_rows']:,}"
            )

    results.sort(key=lambda r: r["chunk_id"])

    X_parts = []
    Y_parts = []

    total_rows = 0
    kept_rows = 0
    bad_events = 0
    empty_events = 0

    for res in results:
        data = np.load(res["outfile"], allow_pickle=True)
        X_parts.append(data["X"])
        Y_parts.append(data["Y"])
        total_rows += int(data["n_total_rows"])
        kept_rows += int(data["n_kept_rows"])
        bad_events += int(data["n_bad_length_events"])
        empty_events += int(data["n_empty_events"])

    X = np.concatenate(X_parts, axis=0) if X_parts else np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
    Y = np.concatenate(Y_parts, axis=0) if Y_parts else np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    np.savez_compressed(
        FINAL_OUTFILE,
        X=X,
        Y=Y,
        feature_names=np.array(FEATURE_BRANCHES, dtype=object),
        label_names=np.array(LABEL_BRANCHES, dtype=object),
    )

    print("\nFinal merged dataset:")
    print("X shape:", X.shape)
    print("Y shape:", Y.shape)
    print(f"Total raw rows: {total_rows:,}")
    print(f"Total kept rows: {kept_rows:,}")
    print(f"Bad-length events skipped: {bad_events}")
    print(f"Empty events skipped: {empty_events}")
    print(f"Saved final dataset to {FINAL_OUTFILE}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Validate a track regression NPZ dataset.

Checks:
  - X and Y exist
  - X and Y are 2D
  - same number of rows
  - expected column counts
  - finite values only
  - optional physics/filter checks:
      * charged only: sim_q != 0
      * nValid > 0
      * eta in [ETA_MIN, ETA_MAX]
  - feature_names / label_names alignment
  - basic per-column summaries
  - sample rows

Usage:
    python validate_track_npz.py path/to/track_data_etaM1to1_charged.npz
"""

from __future__ import annotations

import sys
from pathlib import Path
import numpy as np


# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

DEFAULT_NPZ = "track_data_etaM1to1_charged.npz"

EXPECTED_FEATURE_NAMES = [
    "sim_q",
    "sim_pdgId",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
    "sim_nPixelLay",
    "sim_n3DLay",
    "sim_nTrackerHits",
    "sim_nRecoClusters",
]

EXPECTED_LABEL_NAMES = [
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

ETA_MIN = -1.0
ETA_MAX = 1.0

REQUIRE_CHARGED = True
REQUIRE_NVALID_POSITIVE = True

N_SAMPLE_ROWS = 5


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def fail(msg: str) -> None:
    print(f"[FAIL] {msg}")
    raise SystemExit(1)


def warn(msg: str) -> None:
    print(f"[WARN] {msg}")


def ok(msg: str) -> None:
    print(f"[OK]   {msg}")


def summary_stats(arr: np.ndarray) -> tuple[float, float, float, float]:
    return (
        float(np.min(arr)),
        float(np.max(arr)),
        float(np.mean(arr)),
        float(np.std(arr)),
    )


def print_column_summary(name: str, arr: np.ndarray) -> None:
    mn, mx, mean, std = summary_stats(arr)
    print(
        f"{name:16s} "
        f"min={mn: .6g}  max={mx: .6g}  mean={mean: .6g}  std={std: .6g}"
    )


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    npz_path = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(DEFAULT_NPZ)

    if not npz_path.exists():
        fail(f"File does not exist: {npz_path}")

    print(f"Loading: {npz_path}")
    data = np.load(npz_path, allow_pickle=True)

    required_keys = ["X", "Y"]
    for key in required_keys:
        if key not in data:
            fail(f"Missing required key: {key}")
    ok("Found required arrays X and Y")

    X = data["X"]
    Y = data["Y"]

    print(f"X shape: {X.shape}")
    print(f"Y shape: {Y.shape}")

    # ---- Shape checks ----
    if X.ndim != 2:
        fail(f"X must be 2D, got ndim={X.ndim}")
    if Y.ndim != 2:
        fail(f"Y must be 2D, got ndim={Y.ndim}")
    ok("X and Y are both 2D")

    if X.shape[0] != Y.shape[0]:
        fail(f"Row mismatch: X has {X.shape[0]} rows, Y has {Y.shape[0]} rows")
    ok("X and Y row counts match")

    expected_n_features = len(EXPECTED_FEATURE_NAMES)
    expected_n_labels = len(EXPECTED_LABEL_NAMES)

    if X.shape[1] != expected_n_features:
        fail(
            f"X column mismatch: got {X.shape[1]}, expected {expected_n_features}"
        )
    ok(f"X has expected feature count = {expected_n_features}")

    if Y.shape[1] != expected_n_labels:
        fail(
            f"Y column mismatch: got {Y.shape[1]}, expected {expected_n_labels}"
        )
    ok(f"Y has expected label count = {expected_n_labels}")

    if X.shape[0] == 0:
        fail("Dataset has zero rows")
    ok("Dataset is non-empty")

    # ---- Name checks ----
    feature_names = None
    label_names = None

    if "feature_names" in data:
        feature_names = [str(x) for x in data["feature_names"].tolist()]
        if feature_names != EXPECTED_FEATURE_NAMES:
            warn(f"feature_names in NPZ do not exactly match expected list")
            print("  NPZ feature_names     =", feature_names)
            print("  EXPECTED_FEATURE_NAMES=", EXPECTED_FEATURE_NAMES)
        else:
            ok("feature_names exactly match expected order")
    else:
        warn("No feature_names key found in NPZ")

    if "label_names" in data:
        label_names = [str(x) for x in data["label_names"].tolist()]
        if label_names != EXPECTED_LABEL_NAMES:
            warn("label_names in NPZ do not exactly match expected list")
            print("  NPZ label_names     =", label_names)
            print("  EXPECTED_LABEL_NAMES=", EXPECTED_LABEL_NAMES)
        else:
            ok("label_names exactly match expected order")
    else:
        warn("No label_names key found in NPZ")

    # ---- Finite checks ----
    x_finite = np.isfinite(X)
    y_finite = np.isfinite(Y)

    if not np.all(x_finite):
        bad = np.argwhere(~x_finite)
        fail(f"X contains non-finite values, first few bad indices: {bad[:10].tolist()}")
    ok("All X values are finite")

    if not np.all(y_finite):
        bad = np.argwhere(~y_finite)
        fail(f"Y contains non-finite values, first few bad indices: {bad[:10].tolist()}")
    ok("All Y values are finite")

    # ---- Column index map ----
    feature_index = {name: i for i, name in enumerate(EXPECTED_FEATURE_NAMES)}
    label_index = {name: i for i, name in enumerate(EXPECTED_LABEL_NAMES)}

    # ---- Physics/filter checks ----
    if REQUIRE_CHARGED:
        sim_q = X[:, feature_index["sim_q"]]
        if np.any(sim_q == 0):
            bad_rows = np.where(sim_q == 0)[0][:10]
            fail(f"Found neutral rows even though charged-only filter expected. Example rows: {bad_rows.tolist()}")
        ok("All rows satisfy sim_q != 0")

    if REQUIRE_NVALID_POSITIVE:
        sim_nvalid = X[:, feature_index["sim_nValid"]]
        if np.any(sim_nvalid <= 0):
            bad_rows = np.where(sim_nvalid <= 0)[0][:10]
            fail(f"Found rows with sim_nValid <= 0. Example rows: {bad_rows.tolist()}")
        ok("All rows satisfy sim_nValid > 0")

    sim_pca_eta = Y[:, label_index["sim_pca_eta"]]
    if np.any(sim_pca_eta < ETA_MIN) or np.any(sim_pca_eta > ETA_MAX):
        bad_rows = np.where((sim_pca_eta < ETA_MIN) | (sim_pca_eta > ETA_MAX))[0][:10]
        fail(
            f"Found rows with sim_pca_eta outside [{ETA_MIN}, {ETA_MAX}]. "
            f"Example rows: {bad_rows.tolist()}"
        )
    ok(f"All rows satisfy sim_pca_eta in [{ETA_MIN}, {ETA_MAX}]")

    # ---- Optional sanity checks on label distributions ----
    sim_pca_pt = Y[:, label_index["sim_pca_pt"]]
    if np.any(sim_pca_pt < 0):
        bad_rows = np.where(sim_pca_pt < 0)[0][:10]
        warn(f"Found negative sim_pca_pt rows: {bad_rows.tolist()}")

    # ---- Duplicate / weird-row sanity ----
    if np.all(X == 0):
        fail("All X values are zero")
    ok("X is not entirely zero")

    if np.all(Y == 0):
        fail("All Y values are zero")
    ok("Y is not entirely zero")

    # ---- Summaries ----
    print("\nFeature summaries:")
    for i, name in enumerate(EXPECTED_FEATURE_NAMES):
        print_column_summary(name, X[:, i])

    print("\nLabel summaries:")
    for i, name in enumerate(EXPECTED_LABEL_NAMES):
        print_column_summary(name, Y[:, i])

    # ---- Sample rows ----
    print("\nSample paired rows:")
    rng = np.random.default_rng(0)
    sample_count = min(N_SAMPLE_ROWS, X.shape[0])
    sample_rows = rng.choice(X.shape[0], size=sample_count, replace=False)

    for r in sample_rows:
        print(f"\nRow {int(r)}")
        print("  X:")
        for i, name in enumerate(EXPECTED_FEATURE_NAMES):
            print(f"    {name:16s} = {X[r, i]: .6g}")
        print("  Y:")
        for i, name in enumerate(EXPECTED_LABEL_NAMES):
            print(f"    {name:16s} = {Y[r, i]: .6g}")

    print("\nAll major checks passed.")


if __name__ == "__main__":
    main()

In [ ]:
# Unique lengths??

import ROOT 
import numpy as np

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

BRANCHES = [
    "sim_q",
    "sim_pdgId",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
    "sim_nPixelLay",
    "sim_n3DLay",
    "sim_nTrackerHits",
    "sim_nRecoClusters",
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

def vec_to_np(v):
    return np.asarray(list(v), dtype=np.float32)

f = ROOT.TFile.Open(ROOT_FILE)
t = f.Get(TREE_PATH)

for i_evt in range(min(10, t.GetEntries())):
    t.GetEntry(i_evt)
    print(f"\nEvent {i_evt}")
    lengths = {}
    for name in BRANCHES:
        arr = vec_to_np(getattr(t, name))
        lengths[name] = len(arr)
        print(f"  {name:16s} len = {len(arr)}")
    unique_lengths = sorted(set(lengths.values()))
    print("  unique lengths:", unique_lengths)